In [14]:
import pandas as pd
import re
import numpy as np
import matplotlib.pyplot as plt

In [15]:
df = pd.read_csv(r"C:\Users\Junayed\nyc-311-resolution-time-analysis\data\raw\311_jan_week1_2025.csv")

In [16]:
df.shape

(90565, 10)

In [17]:
df.columns

Index(['unique_key', 'created_date', 'closed_date', 'agency', 'complaint_type',
       'descriptor', 'borough', 'incident_zip', 'status',
       'open_data_channel_type'],
      dtype='str')

In [18]:
df.dtypes

unique_key                  int64
created_date                  str
closed_date                   str
agency                        str
complaint_type                str
descriptor                    str
borough                       str
incident_zip              float64
status                        str
open_data_channel_type        str
dtype: object

In [19]:
df.head(5)

,unique_key,created_date,closed_date,agency,complaint_type,descriptor,borough,incident_zip,status,open_data_channel_type
0,63572086,2025-01-01T00:51:02.000,2025-03-02T00:51:37.000,DOHMH,Smoking or Vaping,Allowed in Smoke Free Area,BRONX,10475.0,Closed,ONLINE
1,63572092,2025-01-01T00:54:26.000,2025-01-01T01:47:36.000,NYPD,Illegal Parking,Parking Permit Improper Use,QUEENS,11373.0,Closed,MOBILE
2,63572113,2025-01-01T01:14:30.000,2025-01-01T04:38:47.000,NYPD,Blocked Driveway,No Access,BROOKLYN,11208.0,Closed,MOBILE
3,63572115,2025-01-01T01:20:52.000,2025-01-01T02:21:14.000,NYPD,Blocked Driveway,No Access,QUEENS,11420.0,Closed,ONLINE
4,63572131,2025-01-01T01:12:27.000,2025-01-01T04:37:23.000,NYPD,Blocked Driveway,No Access,BROOKLYN,11207.0,Closed,MOBILE


In [20]:
df["unique_key"].isna().sum()

np.int64(0)

# Unique key should be unique, let's test this

In [21]:
print("Unique keys: ")
print(f"Total rows: {len(df)}")
print(f"Unique keys: {df["unique_key"].nunique()}")
print(f"Duplicates: {df["unique_key"].duplicated().sum()}")

Unique keys: 
Total rows: 90565
Unique keys: 90565
Duplicates: 0


### 1. Standardize Column Headers to Title Case

Convert all DataFrame column names to title case (e.g., `unique_key` $\rightarrow$ `Unique_Key`) to maintain a clean, consistent presentation across the dataset.

In [22]:
df.columns = df.columns.str.title()

In [23]:
df.head(5)

,Unique_Key,Created_Date,Closed_Date,Agency,Complaint_Type,Descriptor,Borough,Incident_Zip,Status,Open_Data_Channel_Type
0,63572086,2025-01-01T00:51:02.000,2025-03-02T00:51:37.000,DOHMH,Smoking or Vaping,Allowed in Smoke Free Area,BRONX,10475.0,Closed,ONLINE
1,63572092,2025-01-01T00:54:26.000,2025-01-01T01:47:36.000,NYPD,Illegal Parking,Parking Permit Improper Use,QUEENS,11373.0,Closed,MOBILE
2,63572113,2025-01-01T01:14:30.000,2025-01-01T04:38:47.000,NYPD,Blocked Driveway,No Access,BROOKLYN,11208.0,Closed,MOBILE
3,63572115,2025-01-01T01:20:52.000,2025-01-01T02:21:14.000,NYPD,Blocked Driveway,No Access,QUEENS,11420.0,Closed,ONLINE
4,63572131,2025-01-01T01:12:27.000,2025-01-01T04:37:23.000,NYPD,Blocked Driveway,No Access,BROOKLYN,11207.0,Closed,MOBILE


### Inspect Ticket Status Distribution

Check the frequency distribution of complaint resolution statuses (`df["Status"]`) to understand how many requests are closed, pending, or active within this period.

In [24]:
df["Status"].value_counts()

Status
Closed         90103
In Progress      193
Open             121
Assigned          62
Pending           37
Unspecified       29
Started           20
Name: count, dtype: int64

### Export and Remove Non-Closed Service Requests

Audit, export, and remove all records where `Status` is not `"Closed"`.

**Process:**
* **Export Dropped Data:** Save records with incomplete resolution lifecycles to `Dropped_data.csv` for documentation and auditing.
* **Filter Main Dataset:** Drop these rows by index to ensure subsequent analyses rely exclusively on fully resolved requests with valid `Closed_Date` values.
* **Verification:** Display the count of dropped rows and confirm that only `"Closed"` entries remain.

In [26]:
## keeping the dropping rows
rows_to_drop = df[df["Status"] != "Closed"]
rows_to_drop.to_csv("Dropped_data.csv", index=False)


rows_to_drop = df[df["Status"] != "Closed"].index

total_rows_drop = len(rows_to_drop)
print(f"{total_rows_drop} rows are being dropped.")

df = df.drop(index=rows_to_drop)
df["Status"].value_counts()

462 rows are being dropped.


Status
Closed    90103
Name: count, dtype: int64

### Parse Request Creation Timestamps

Convert the `Created_Date` column from string/object format into standard `datetime64[ns]` objects.

* **Format Handling:** Setting `format="mixed"` accommodates ISO 8601 formatting and potential variations in timestamp representations across rows.
* **Error Handling:** Using `errors='coerce'` converts unparseable strings into `NaT` (Not a Time) rather than interrupting execution.
* **Validation:** Preview the first 10 converted values to verify parsing accuracy.

In [28]:
df["Created_Date"] = pd.to_datetime(df["Created_Date"], format="mixed", errors='coerce')
df["Created_Date"].head(10)

0   2025-01-01 00:51:02
1   2025-01-01 00:54:26
2   2025-01-01 01:14:30
3   2025-01-01 01:20:52
4   2025-01-01 01:12:27
5   2025-01-01 00:07:36
6   2025-01-01 00:19:00
7   2025-01-01 00:11:45
8   2025-01-01 00:14:54
9   2025-01-01 00:17:41
Name: Created_Date, dtype: datetime64[us]

### Parse Request Resolution Timestamps

Convert the `Closed_Date` column from string/object format into `datetime64[ns]` objects to enable downstream time-to-close calculations.

* **Format Handling:** Uses `format="mixed"` to reliably parse standard ISO 8601 timestamps and variations across rows.
* **Error Handling:** Applies `errors='coerce'` to safeguard against corrupt date strings by converting invalid entries into `NaT`.
* **Validation:** Preview the first 10 converted values to verify clean date parsing.

In [29]:
df["Closed_Date"] = pd.to_datetime(df["Closed_Date"], format='mixed', errors='coerce')
df["Closed_Date"].head(10)

0   2025-03-02 00:51:37
1   2025-01-01 01:47:36
2   2025-01-01 04:38:47
3   2025-01-01 02:21:14
4   2025-01-01 04:37:23
5   2025-01-01 01:33:12
6   2025-01-01 00:19:00
7   2025-01-01 01:21:57
8   2025-01-01 01:19:02
9   2025-01-01 00:30:38
Name: Closed_Date, dtype: datetime64[us]

### Check for Missing Resolution Timestamps

Verify data completeness by calculating the total number of missing (`NaT` / `NaN`) values in `Closed_Date` following the datetime conversion. 

* Even within records marked as `"Closed"`, missing timestamps (e.g., records closed administratively without an explicit resolution timestamp or unparseable date strings) cannot be used in duration calculations.
* This count determines whether further row pruning is required before calculating turnaround times.

In [27]:
df["Closed_Date"].isna().sum()

np.int64(230)

### Impute Missing Resolution Dates

Handle the 230 records flagged as `"Closed"` that are missing an explicit `Closed_Date` timestamp by applying an end-of-week (EOW) business assumption:

* **Audit Annotation:** Update the `Descriptor` field to `"No closed date, I assumed they were closed at EOW."` to ensure transparency and traceability for downstream consumers.
* **Timestamp Imputation:** Impute missing `Closed_Date` entries with the end of the analysis period (`2025-01-07 23:59:59`).
* **Validation:** Re-evaluate `isna().sum()` on `Closed_Date` to verify zero null timestamps remain.

In [30]:
df.loc[df["Closed_Date"].isna(), "Descriptor"] = "No closed date, I assumed they were closed at EOW."

In [33]:
df.loc[df["Closed_Date"].isna(), "Closed_Date"] = pd.to_datetime("2025-01-07 23:59:59")
df["Closed_Date"].isna().sum()

np.int64(0)

### Compute Ticket Resolution Time in Days

Engineer a new metric, `Resolution_Days`, to quantify how long each service request took to resolve:

* **Calculation:** Subtract complaint creation time (`Created_Date`) from resolution time (`Closed_Date`), extracting the integer component via `.dt.days`.
* **Metric Definition:** Captures the full elapsed calendar days required by agencies to address and close each ticket.
* **Preview:** Display the first 10 computed values to confirm proper feature derivation.

In [34]:
df["Resolution_Days"] = (df["Closed_Date"] - df["Created_Date"]).dt.days
df["Resolution_Days"].head(10)

0    60
1     0
2     0
3     0
4     0
5     0
6     0
7     0
8     0
9     0
Name: Resolution_Days, dtype: int64

### Analyze Same-Day Resolution Volume

Quantify how many service requests were resolved within 24 hours of submission (`Resolution_Days == 0`).

* **Metric Finding:** 56,320 tickets were closed on the same calendar day as intake.
* **Operational Significance:** A large cluster of same-day closures indicates high throughput for immediate or rapid-dispatch complaint types (such as parking enforcement or immediate field inspections).

In [36]:
print(f"Same-day reolutions(0 days): {(df["Resolution_Days"] == 0).sum()}")

Same-day reolutions(0 days): 56320
